# 22 - Advanced Gates Showcase

Explore all 50+ gates available in QuantSDK.

**Concepts:** Full gate library, rotation gates, controlled gates, multi-qubit gates

In [ ]:
import quantsdk as qs
import math

## Single-Qubit Gates

In [ ]:
# Pauli gates
for gate_name in ['X', 'Y', 'Z']:
    circuit = qs.Circuit(1, name=gate_name)
    getattr(circuit, gate_name.lower())(0)
    circuit.measure(0)
    result = qs.run(circuit, shots=100, seed=42)
    print(f"{gate_name} gate: {result.counts}")

In [ ]:
# Phase gates: S, Sdg, T, Tdg
for gate in ['s', 'sdg', 't', 'tdg']:
    circuit = qs.Circuit(1, name=gate)
    circuit.h(0)  # Create superposition first
    getattr(circuit, gate)(0)
    circuit.h(0)
    circuit.measure(0)
    result = qs.run(circuit, shots=1000, seed=42)
    print(f"{gate:4s}: {result.counts}")

In [ ]:
# SX gate (sqrt-X)
circuit = qs.Circuit(1)
circuit.sx(0).sx(0)  # SX^2 = X
circuit.measure(0)
result = qs.run(circuit, shots=100, seed=42)
print(f"SX*SX = X: {result.counts} (should be all 1s)")

# Identity
circuit = qs.Circuit(1)
circuit.i(0)
circuit.measure(0)
result = qs.run(circuit, shots=100, seed=42)
print(f"I gate: {result.counts} (should be all 0s)")

## Rotation Gates

In [ ]:
# RX, RY, RZ at pi
for gate in ['rx', 'ry', 'rz']:
    circuit = qs.Circuit(1, name=gate)
    getattr(circuit, gate)(0, math.pi)
    circuit.measure(0)
    result = qs.run(circuit, shots=100, seed=42)
    print(f"{gate}(pi): {result.counts}")

print()

# U3 gate (most general single-qubit)
circuit = qs.Circuit(1)
circuit.u3(0, math.pi, 0, math.pi)  # Equivalent to X
circuit.measure(0)
result = qs.run(circuit, shots=100, seed=42)
print(f"U3(pi,0,pi) = X: {result.counts}")

# Phase gate
circuit = qs.Circuit(1)
circuit.h(0).phase(0, math.pi / 4).h(0)
circuit.measure(0)
result = qs.run(circuit, shots=1000, seed=42)
print(f"Phase(pi/4): {result.counts}")

## Controlled Gates

In [ ]:
# CX, CY, CZ
for gate in ['cx', 'cy', 'cz']:
    circuit = qs.Circuit(2)
    circuit.x(0)  # Control = |1>
    getattr(circuit, gate)(0, 1)
    circuit.measure_all()
    result = qs.run(circuit, shots=100, seed=42)
    print(f"{gate.upper()} (ctrl=1): {result.most_likely}")

print()

# CH (controlled-Hadamard)
circuit = qs.Circuit(2)
circuit.x(0).ch(0, 1).measure_all()
result = qs.run(circuit, shots=1000, seed=42)
print(f"CH (ctrl=1): {result.counts}")

# CS, CSdg
for gate in ['cs', 'csdg']:
    circuit = qs.Circuit(2)
    circuit.h(1).x(0)
    getattr(circuit, gate)(0, 1)
    circuit.h(1).measure_all()
    result = qs.run(circuit, shots=1000, seed=42)
    print(f"{gate.upper()}: {result.counts}")

In [ ]:
# Controlled rotations: CRX, CRY, CRZ, CP
for gate in ['crx', 'cry', 'crz', 'cp']:
    circuit = qs.Circuit(2)
    circuit.x(0)  # Control = 1
    getattr(circuit, gate)(0, 1, math.pi / 2)
    circuit.measure_all()
    result = qs.run(circuit, shots=1000, seed=42)
    print(f"{gate.upper()}(pi/2): {result.counts}")

## Two-Qubit Special Gates

In [ ]:
# SWAP
circuit = qs.Circuit(2)
circuit.x(0).swap(0, 1).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"SWAP |10> -> {result.most_likely}")

# iSWAP
circuit = qs.Circuit(2)
circuit.x(0).iswap(0, 1).measure_all()
result = qs.run(circuit, shots=1000, seed=42)
print(f"iSWAP |10>: {result.counts}")

# DCX (double CNOT)
circuit = qs.Circuit(2)
circuit.x(0).dcx(0, 1).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"DCX |10>: {result.most_likely}")

# ECR gate
circuit = qs.Circuit(2)
circuit.ecr(0, 1).measure_all()
result = qs.run(circuit, shots=1000, seed=42)
print(f"ECR |00>: {result.counts}")

In [ ]:
# Ising coupling gates: RZZ, RXX, RYY, RZX
for gate in ['rzz', 'rxx', 'ryy', 'rzx']:
    circuit = qs.Circuit(2)
    getattr(circuit, gate)(0, 1, math.pi / 4)
    circuit.measure_all()
    result = qs.run(circuit, shots=1000, seed=42)
    print(f"{gate.upper()}(pi/4): {result.counts}")

## Three-Qubit Gates

In [ ]:
# Toffoli (CCX)
circuit = qs.Circuit(3)
circuit.x(0).x(1).ccx(0, 1, 2).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"Toffoli |110> -> {result.most_likely} (target flipped)")

# CCZ
circuit = qs.Circuit(3)
circuit.x(0).x(1).h(2).ccz(0, 1, 2).h(2).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"CCZ equiv: {result.most_likely}")

# Fredkin (CSWAP)
circuit = qs.Circuit(3)
circuit.x(0).x(1).cswap(0, 1, 2).measure_all()  # Swap q1,q2 when q0=1
result = qs.run(circuit, shots=100, seed=42)
print(f"Fredkin |110> -> {result.most_likely} (q1,q2 swapped)")

## Gate Count Summary

In [ ]:
from quantsdk.gates import GATE_MAP

print(f"Total gates in GATE_MAP: {len(GATE_MAP)}")
print(f"\nAll gate names:")
for i, name in enumerate(sorted(GATE_MAP.keys())):
    print(f"  {name:12s}", end="")
    if (i + 1) % 5 == 0:
        print()